# Week 94, Day 5: Finding a first issue

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w094d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('issues.py', 'w', encoding='utf-8') as f:
    f.write('FENCE = "`" * 3\n\nISSUES = [\n    {"number": 140101, "title": "textwrap.shorten drops a trailing word", "labels": ["type-bug", "stdlib", "easy"],\n     "opened": "2026-10-02", "comments": 3, "assignees": [], "linked_prs": [],\n     "body": f"On 3.14.0:\\n{FENCE}\\n>>> textwrap.shorten(\'a b c\', 4)\\n{FENCE}\\nExpected \'a b\' but got \'a\'."},\n    {"number": 140115, "title": "Clarify json.dumps sort_keys in the docs", "labels": ["docs", "easy"],\n     "opened": "2026-09-14", "comments": 1, "assignees": [], "linked_prs": [], "body": "The docs don\'t say what happens with mixed key types."},\n    {"number": 139870, "title": "Rewrite the import system in Rust", "labels": ["type-feature", "needs decision"],\n     "opened": "2026-06-01", "comments": 48, "assignees": [], "linked_prs": [], "body": "A proposal."},\n    {"number": 140002, "title": "Typo in the tutorial\'s classes chapter", "labels": ["docs", "easy"],\n     "opened": "2026-08-30", "comments": 2, "assignees": [], "linked_prs": [140010], "body": "\'the the\' in section 9.3."},\n    {"number": 120455, "title": "Add tests for calendar.TextCalendar.formatyear", "labels": ["tests", "easy"],\n     "opened": "2024-05-11", "comments": 6, "assignees": [], "linked_prs": [], "body": "Some arguments have no tests."},\n    {"number": 140130, "title": "Crash in a C extension with a huge list", "labels": ["type-crash", "C-API"],\n     "opened": "2026-10-20", "comments": 9, "assignees": ["core-dev-1"], "linked_prs": [], "body": "Segfault, see the attached script."},\n    {"number": 140144, "title": "difflib.get_close_matches ignores cutoff=0", "labels": ["type-bug", "stdlib"],\n     "opened": "2026-11-01", "comments": 0, "assignees": [], "linked_prs": [], "body": "It returns nothing."},\n]\n')

**Predict before you run:** what will this print?



In [ ]:
from issues import ISSUES

untaken = [i["number"] for i in ISSUES if not i["assignees"] and not i["linked_prs"]]
print(len(ISSUES), untaken)

> **Key idea.** A good first issue is small, wanted, untaken (no assignee, no open linked PR), reproducible, and not years old; docs and tests are ideal. Search with qualifiers (`is:open label:easy no:assignee "word"`). Read everything, reproduce on `main`, and comment only with something useful and a plan; the pull request is the claim. Be patient and polite: maintainers are mostly volunteers.

### Exercise: Searching issues

Write `search_query(repo, labels=(), words=(), unassigned=True, exclude=())`: return a GitHub search string: `repo:REPO is:issue is:open`, then `label:L` for each label (quoted as `label:"L"` if it contains a space), `-label:L` for each excluded label (quoted the same way), `no:assignee` if `unassigned`, and each word in double quotes, separated by spaces.

In [ ]:
def search_query(repo, labels=(), words=(), unassigned=True, exclude=()):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_query():\n    \"GitHub's qualifiers\"\n    assert search_query(\"python/cpython\", [\"easy\", \"docs\"], [\"json\"]) == 'repo:python/cpython is:issue is:open label:easy label:docs no:assignee \"json\"'\n    assert search_query(\"pallets/flask\", [\"good first issue\"], exclude=[\"needs decision\"], unassigned=False) == (\n        'repo:pallets/flask is:issue is:open label:\"good first issue\" -label:\"needs decision\"')\n    assert search_query(\"a/b\") == \"repo:a/b is:issue is:open no:assignee\"")

### Exercise: How suitable is it?

Write `suitability(issue, today)`: return `None` if the issue has assignees or linked pull requests. Otherwise start at 0 and add 3 for `easy` or `good first issue`; 2 for `docs` or `tests`; 1 for `type-bug`; subtract 3 for `needs decision`; 2 if it has more than 20 comments; 1 if it was opened more than 730 days before `today` (dates are `"YYYY-MM-DD"`); and 2 for `type-crash` or `C-API`. Then `rank(issues, today)`: the suitable issues as `"gh-N (SCORE): TITLE"`, highest score first, then newest (larger number) first.

In [ ]:
from datetime import date

def suitability(issue, today):
    ___

def rank(issues, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from issues import ISSUES\n\ndef test_scores():\n    \"small, wanted, untaken\"\n    scores = {i[\"number\"]: suitability(i, \"2026-11-20\") for i in ISSUES}\n    assert scores == {140101: 4, 140115: 5, 139870: -5, 140002: None, 120455: 4, 140130: None, 140144: 1}\n\ndef test_rank():\n    \"best first\"\n    assert rank(ISSUES, \"2026-11-20\") == [\n        \"gh-140115 (5): Clarify json.dumps sort_keys in the docs\",\n        \"gh-140101 (4): textwrap.shorten drops a trailing word\",\n        \"gh-120455 (4): Add tests for calendar.TextCalendar.formatyear\",\n        \"gh-140144 (1): difflib.get_close_matches ignores cutoff=0\",\n        \"gh-139870 (-5): Rewrite the import system in Rust\",\n    ]")

### Exercise: What's the report missing?

Write `missing_info(body)`: return, in this order, `"version"` if there's no version number (digits, a dot, digits), `"reproduction"` if there's neither a code block (three backticks) nor a line starting with `>>>`, and `"expected and actual"` unless the text (ignoring case) mentions `expected` and also one of `actual`, `got`, `instead` or `but`.

In [ ]:
import re

def missing_info(body):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from issues import ISSUES\n\ndef test_missing():\n    \"what to ask for, or find yourself\"\n    by_number = {i[\"number\"]: i[\"body\"] for i in ISSUES}\n    assert missing_info(by_number[140101]) == []\n    assert missing_info(by_number[140144]) == [\"version\", \"reproduction\", \"expected and actual\"]\n    assert missing_info(\">>> difflib.get_close_matches('a', ['a'], cutoff=0)\\nExpected ['a'] but got []\") == [\"version\"]")

### Exercise: A comment worth reading

Write `comment_problems(comment)`: report, in this order, `"asks to be assigned"` (the text, ignoring case, contains `assign me` or `assign this to me`), `"mentions people"` (an `@` followed by a letter), `"pushes for speed"` (`asap`, `urgent` or `please hurry`, as words or phrases, ignoring case), and `"adds nothing new"` unless it contains a code block, a line starting with `>>>`, a version number, or the word `reproduce`, `reproduced`, `bisect` or `bisected`.

In [ ]:
import re

def comment_problems(comment):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_comments():\n    \"useful, patient, no pings\"\n    good = \"I reproduced this on main (3.15.0a1+): `shorten('a b c', 4)` returns 'a'. I plan to open a pull request with a test.\"\n    assert comment_problems(good) == []\n    bad = \"Hi @maintainer, please assign me! I need this ASAP.\"\n    assert comment_problems(bad) == [\"asks to be assigned\", \"mentions people\", \"pushes for speed\", \"adds nothing new\"]\n    assert comment_problems(\"Can I work on this? Email me at a@b.org\") == [\"adds nothing new\"]")

### Bug hunt: Every issue looks taken

An empty list of pull requests isn't `None`, so this treats every issue as taken and prints `[]`. Check for an empty list so it prints `[12]`.

In [ ]:
issues = [{"number": 12, "linked_prs": []}, {"number": 15, "linked_prs": [16]}]
print([i["number"] for i in issues if i["linked_prs"] is None])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"empty lists are falsy\"\n    expect_output(\"[12]\")")

### Bug hunt: Old issues counted as new

The comparison is the wrong way round, so the issue opened in 2024 isn't flagged as old and the 2026 one is: it prints `[140101]`. Fix it so it prints `[120455]`.

In [ ]:
from datetime import date

today = date(2026, 11, 20)
opened = {120455: date(2024, 5, 11), 140101: date(2026, 10, 2)}
print([n for n, d in opened.items() if (today - d).days < 730])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"more than two years\"\n    expect_output(\"[120455]\")")

### Puzzle: Ranking the suitable issues

Arrange `rank`: score each issue, drop the unsuitable, sort by score then newest, and format. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the label CPython uses for issues suited to newcomers
q = ___   # the search qualifier that skips assigned issues
r = ___   # True or False: you must be assigned before opening a pull request to CPython
s = ___   # suitability of an untaken issue labelled only "docs", opened last week with 1 comment
t = ___   # what claims an issue in CPython, in two words

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"easy\" and q == \"no:assignee\" and r is False, \"p, q and r\"\n    assert s == 2 and t == \"pull request\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: choose an issue

Write `choose(issues, today, skills)` with today's helpers (given). `skills` is a set of labels you're comfortable with (such as `{"docs", "stdlib"}`). Each suitable issue's **fit** is its suitability plus 1 for each of its labels in `skills`. Return `(lines, recommendation)`: one line per suitable issue, best fit first (ties: newest first), `"gh-N fit F: TITLE"`, followed (on the same line) by ` [missing: A, B]` if `missing_info` finds anything in its body. `recommendation` is `"start with gh-N: reproduce it on main"` for the best fit if it's missing nothing, `"start with gh-N: first find the A and B"` (the missing items joined by `" and "`) if it is, or `"no suitable issue"`.

In [ ]:
import re
from datetime import date

def suitability(issue, today):
    if issue["assignees"] or issue["linked_prs"]:
        return None
    labels = set(issue["labels"])
    score = 3 * bool(labels & {"easy", "good first issue"}) + 2 * bool(labels & {"docs", "tests"}) + ("type-bug" in labels)
    score -= 3 * ("needs decision" in labels) + 2 * (issue["comments"] > 20) + 2 * bool(labels & {"type-crash", "C-API"})
    score -= (date.fromisoformat(today) - date.fromisoformat(issue["opened"])).days > 730
    return score

def missing_info(body):
    lower = body.lower()
    missing = []
    if not re.search(r"\d+\.\d+", body):
        missing.append("version")
    if "`" * 3 not in body and not any(line.lstrip().startswith(">>>") for line in body.splitlines()):
        missing.append("reproduction")
    if not ("expected" in lower and re.search(r"\b(actual|got|instead|but)\b", lower)):
        missing.append("expected and actual")
    return missing

def choose(issues, today, skills):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from issues import ISSUES\n\ndef test_docs_person():\n    \"someone who likes writing\"\n    lines, recommendation = choose(ISSUES, \"2026-11-20\", {\"docs\"})\n    assert lines == [\n        \"gh-140115 fit 6: Clarify json.dumps sort_keys in the docs [missing: version, reproduction, expected and actual]\",\n        \"gh-140101 fit 4: textwrap.shorten drops a trailing word\",\n        \"gh-120455 fit 4: Add tests for calendar.TextCalendar.formatyear [missing: version, reproduction, expected and actual]\",\n        \"gh-140144 fit 1: difflib.get_close_matches ignores cutoff=0 [missing: version, reproduction, expected and actual]\",\n        \"gh-139870 fit -5: Rewrite the import system in Rust [missing: version, reproduction, expected and actual]\",\n    ]\n    assert recommendation == \"start with gh-140115: first find the version and reproduction and expected and actual\"\n\ndef test_stdlib_person():\n    \"someone who likes code\"\n    lines, recommendation = choose(ISSUES, \"2026-11-20\", {\"stdlib\", \"type-bug\"})\n    assert lines[0] == \"gh-140101 fit 6: textwrap.shorten drops a trailing word\"\n    assert recommendation == \"start with gh-140101: reproduce it on main\"\n    assert choose([i for i in ISSUES if i[\"assignees\"] or i[\"linked_prs\"]], \"2026-11-20\", set()) == ([], \"no suitable issue\")")

## Warm-up

Two of the seven are already being worked on: one is assigned to a core developer, and one has an open pull request. Working on those wastes your time and someone else's. Checking takes a minute.

## What makes a good first issue

- **Small**: a documentation fix, a missing test, a bug in a pure-Python module with a clear cause. Not a crash in the C core, not a redesign.
- **Wanted**: a maintainer has agreed, or the label says so (`easy`, `docs`). An issue labelled `needs decision`, or with forty comments of argument, isn't ready for anyone, least of all a newcomer.
- **Not taken**: no assignee and no linked open pull request. If someone said "I'll do this" months ago and nothing happened, it's usually fine to ask politely whether they're still working on it.
- **Reproducible**: the report has the version, a short example, and what was expected against what happened. If it doesn't, finding those is itself a useful contribution.
- **Not too old**: issues open for years are often open because they're harder than they look.

Documentation and tests are excellent first contributions: they're wanted, reviewed quickly, and teach you the workflow without the risk of breaking anything.

## Searching

GitHub's issue search takes **qualifiers**: `repo:python/cpython is:issue is:open label:easy label:docs "json"`. Labels with spaces are quoted (`label:"good first issue"`, the label many other projects use), and `-label:x` excludes one. Add `no:assignee` to skip assigned issues. Save a search you like and check it every few days: good first issues go quickly.

## Before you start

Read the whole issue, every comment, and any linked pull requests. Reproduce the problem on your own build of `main` (Day 4). Then, if it helps, leave a short comment: what you found (a reproduction, the version it started in) and that you plan to open a pull request. CPython doesn't need you to be assigned: the pull request is the claim. Don't ask to be assigned, don't mention people to get their attention, and don't push for speed: maintainers are mostly volunteers, and a pull request that waits a week isn't being ignored.

## Practice

## Quiz

## Daily challenge

## Your turn: choose one

1. **Search** CPython's open issues with `label:easy no:assignee` (and `label:docs` or `label:tests`), and save the search.
2. **Score five** of them with your own `suitability`, reading every comment and linked pull request.
3. **Reproduce the best one** on your own build of `main`. If the report is missing anything, find it.
4. **Comment** only if you've found something useful, with your plan. Then start: next week is the fix, the test and the pull request.

## Before you go

Tomorrow reviews the week, from reading codebases to choosing an issue.